<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Final Project · My Fund Dashboard

Your fund has been live on the class leaderboard since Studio 5. This notebook turns its real record into a one-page
dashboard for your report:

1. **The race:** your fund, its no-rules twin and the S&P 500, every trading day, with your changes marked.
2. **The gaps:** your fund minus the S&P 500, and your rules minus no rules, in dollars. Where did you win or lose
   your lead?
3. **What drove it:** how many dollars each stock added or took away.
4. **How you managed it:** every change with its reason, the dollars traded, and what a 0.1% trading cost would have
   cost you. (The leaderboard itself does not charge trading costs, so this is the honest version.)

Run the cells top to bottom. At the end you get `my_fund_dashboard.png` to put in your report.

## Import modules

In [ ]:
%matplotlib inline
from pylab import *
import json
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

## Step 1 · Your private record

Your Studio 5 or Studio 6 receipt printed **your private record** (a link ending in `.json`). Paste it below. It holds
every entry you sent, with the stocks and shares, and your fund's value every day. Keep the link to yourself.

Lost it? Send your fund again from Studio 6 (the receipt prints it), or email Dr. Dave.

In [ ]:
record_link = ""     # paste the link from your receipt, between the quotes

settings = {
    'trading_cost': 0.001,      # 0.1% of every dollar bought or sold (for the "what it would have cost" numbers)
    'top_stocks': 8,            # how many of the biggest winners and losers to show
}

if record_link.strip() == "":
    print("Paste your record link first (from your leaderboard receipt).")
else:
    record = requests.get(record_link.strip(), timeout=30).json()
    print("fund:", record['nickname'], "|", record['theme'])
    print("entries:", len(record['entries']), "| days on the board:", len(record['values']['dates']),
          "| prices through the close of", record['last_close'])

## Step 2 · Your fund, day by day

The leaderboard's numbers, exactly as the class sees them: \$10,000 at the first close after your first entry,
held, and carried forward through every change.

In [ ]:
values = pd.DataFrame({'with your rules': record['values']['with_rules'], 'no rules': record['values']['no_rules']},
                      index=pd.to_datetime(record['values']['dates']))
spy = pd.Series(record['spy']['values'], index=pd.to_datetime(record['spy']['dates']))
values['S&P 500 (SPY)'] = spy.reindex(values.index).ffill()

# the changes that took effect (two entries before the same close: the later one counts)
changes = {}
for entry in record['entries']:
    if entry['counts_from'] is not None:
        changes[pd.Timestamp(entry['counts_from'])] = entry
change_days = sorted(changes)

print("started", values.index[0].date(), "| changes that took effect:", len(change_days) - 1)
values.tail()

## Step 3 · What drove it, and what trading would have cost

Between two changes your fund is bought and held, so each stock's dollars grow with its own price. This step
re-builds that from real prices, stock by stock:

* **added (\$)** is how many dollars each stock added to (or took from) your fund,
* at every change, **traded (\$)** is how much you sold and bought to move from the old mix to the new one.

The added dollars should add up to your fund's gain on the leaderboard (the check at the bottom).

In [ ]:
tickers = []
for day in change_days:
    for t in changes[day]['portfolios']['with_rules']:
        if t not in tickers:
            tickers.append(t)
prices = yf.download(tickers, start=str(values.index[0].date()), interval='1d', auto_adjust=True, progress=False)['Close']
if isinstance(prices, pd.Series):
    prices = prices.to_frame(name=tickers[0])
prices.index = pd.to_datetime(prices.index).tz_localize(None).normalize()
prices = prices.reindex(values.index).ffill()

added = {}
for t in tickers:
    added[t] = 0.0
log = []
for k in range(len(change_days)):
    day = change_days[k]
    if k + 1 < len(change_days):
        end = change_days[k + 1]
    else:
        end = values.index[-1]
    weights = changes[day]['portfolios']['with_rules']
    start_value = values.loc[day, 'with your rules']
    traded = start_value
    if k > 0:
        # what we held just before the change (the old mix after it drifted) vs the new mix
        old = changes[change_days[k - 1]]['portfolios']['with_rules']
        old_start = change_days[k - 1]
        old_value = values.loc[old_start, 'with your rules']
        held = {}
        for t in old:
            if t in prices.columns and not np.isnan(prices.loc[old_start, t]):
                held[t] = old_value * old[t] * prices.loc[day, t] / prices.loc[old_start, t]
            else:
                held[t] = old_value * old[t]
        traded = 0.0
        names = list(set(list(old) + list(weights)))
        for t in names:
            traded += abs(start_value * weights.get(t, 0) - held.get(t, 0))
    for t in weights:
        if t in prices.columns and not np.isnan(prices.loc[day, t]):
            added[t] += start_value * weights[t] * (prices.loc[end, t] / prices.loc[day, t] - 1)
    log.append({'from the close on': str(day.date()), 'reason': changes[day]['reason'],
                'stocks': len(weights), 'traded ($)': round(traded, 0),
                'cost at 0.1% ($)': round(settings['trading_cost'] * traded, 2)})

added = pd.Series(added).sort_values()
log = pd.DataFrame(log)
gain = values['with your rules'].iloc[-1] - 10000
print("leaderboard gain: $%.2f | sum of the stocks' added dollars: $%.2f" % (gain, added.sum()))
display(log)

## Step 4 · The dashboard

One figure, four panels. It saves as `my_fund_dashboard.png` (left sidebar, Files; or the download below).

In [ ]:
fund = values['with your rules']
twin = values['no rules']
market = values['S&P 500 (SPY)']
total_cost = log['cost at 0.1% ($)'].sum()
worst_drop = 100 * (1 - fund / fund.cummax()).max()

fig = figure(figsize=(15, 10))
fig.suptitle("%s: %s" % (record['nickname'], record['theme']), fontsize=16, fontweight='bold')

# 1. the race
panel = fig.add_subplot(2, 2, 1)
panel.plot(fund.index, fund.values, color='tab:orange', linewidth=2.5, label='my fund (with my rules)')
panel.plot(twin.index, twin.values, color='gray', linewidth=1.6, label='its no-rules twin')
panel.plot(market.index, market.values, color='black', linewidth=1.6, label='S&P 500 (SPY)')
for day in change_days[1:]:
    panel.axvline(day, color='tab:orange', linestyle=':', linewidth=1)
panel.axhline(10000, color='lightgray', linestyle='--')
panel.set_title('1. The race (dotted lines = my changes)')
panel.tick_params(axis='x', rotation=30)
panel.set_ylabel('value of $10,000')
panel.legend(fontsize=8)

# 2. the gaps
panel = fig.add_subplot(2, 2, 2)
panel.plot(fund.index, (fund - market).values, color='black', linewidth=2, label='my fund minus the S&P 500')
panel.plot(fund.index, (fund - twin).values, color='tab:orange', linewidth=2, label='my rules minus no rules')
panel.axhline(0, color='lightgray')
panel.set_title('2. The gaps, in dollars (above 0 = ahead)')
panel.tick_params(axis='x', rotation=30)
panel.legend(fontsize=8)

# 3. what drove it
panel = fig.add_subplot(2, 2, 3)
shown = pd.concat([added.head(settings['top_stocks']), added.tail(settings['top_stocks'])])
shown = shown[~shown.index.duplicated()]
colors = []
for t in shown.index:
    if shown[t] >= 0:
        colors.append('tab:green')
    else:
        colors.append('tab:red')
panel.barh(shown.index, shown.values, color=colors)
panel.axvline(0, color='black', linewidth=0.8)
panel.set_title('3. What drove it: dollars each stock added or took away')

# 4. the scorecard
panel = fig.add_subplot(2, 2, 4)
panel.axis('off')
lines = [
    "Days on the board:               %d  (since %s)" % (len(fund), fund.index[0].date()),
    "My fund:                         $%s  (%+.2f%%)" % ("{:,.0f}".format(fund.iloc[-1]), 100 * (fund.iloc[-1] / 10000 - 1)),
    "Its no-rules twin:               $%s  (%+.2f%%)" % ("{:,.0f}".format(twin.iloc[-1]), 100 * (twin.iloc[-1] / 10000 - 1)),
    "S&P 500, same days:              $%s  (%+.2f%%)" % ("{:,.0f}".format(market.iloc[-1]), 100 * (market.iloc[-1] / 10000 - 1)),
    "Worst drop from a high:          %.2f%%" % worst_drop,
    "Changes that took effect:        %d" % (len(change_days) - 1),
    "Dollars traded (incl. first buy): $%s" % "{:,.0f}".format(log['traded ($)'].sum()),
    "A 0.1%% trading cost would take:  $%.2f" % total_cost,
    "Biggest help:  %s (%+.0f dollars)" % (added.index[-1], added.iloc[-1]),
    "Biggest drag:  %s (%+.0f dollars)" % (added.index[0], added.iloc[0]),
    "",
    "Latest reason: " + str(log['reason'].iloc[-1])[:60],
]
for k in range(len(lines)):
    panel.text(0.0, 1 - 0.08 * k, lines[k], fontsize=11, family='monospace', va='top', transform=panel.transAxes)
panel.set_title('4. The scorecard', loc='left')

fig.tight_layout()
fig.savefig('my_fund_dashboard.png', dpi=120)
show()
print("saved my_fund_dashboard.png")

In [ ]:
# download the dashboard (Colab)
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_fund_dashboard.png')

## For your report

Use the dashboard and the numbers above, together with your Studio 6 backtest, to answer:

1. **Rules vs no rules.** Live (panel 2) and in your Studio 6 backtest: did your rules help or hurt, and why?
2. **Rebalancing.** What schedule did you choose in Studio 6, did you stick to it (your changes in Step 3), and what
   did it trade and cost?
3. **Forensics.** Which stocks decided your result (panel 3)? Find one day in panel 2 where a gap jumped, and explain
   it.
4. **What you would change** for 2027, and why.